# Topic Modelling

This notebook covers Topic Modelling, a technique for discovering abstract topics in a collection of documents. Topic modelling is an unsupervised learning method that identifies hidden semantic patterns in text corpora.

Two popular algorithms are covered:
- **LDA (Latent Dirichlet Allocation)**: The most widely used topic model
- **NMF (Non-negative Matrix Factorisation)**: An alternative approach that often produces more interpretable topics

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

## 1. Understanding Topic Modelling

Topic modelling assumes that each document is a mixture of topics, and each topic is a distribution over words. The goal is to discover these latent topics from the document collection.

**Key concepts:**
- **Document**: A single text (can be a sentence, paragraph, or article)
- **Topic**: A cluster of related words that represent a theme
- **Word-Topic Distribution**: Probability of each word belonging to each topic
- **Document-Topic Distribution**: Proportion of each topic in each document

In [ ]:
# Create a sample corpus with distinct topics
corpus = [
    # Technology topic
    "Machine learning and artificial intelligence are transforming technology.",
    "Deep learning uses neural networks to learn patterns from data.",
    "Python is a popular programming language for data science.",
    "Neural networks are inspired by the human brain.",
    "TensorFlow and PyTorch are popular deep learning frameworks.",
    
    # Sports topic
    "Football is the most popular sport in the world.",
    "The football team won the championship last night.",
    "Cricket is played with a bat and ball in many countries.",
    "The tennis player served an ace to win the match.",
    "Basketball players score points by shooting the ball into the hoop.",
    
    # Food topic
    "Italian cuisine includes pasta, pizza, and risotto.",
    "The restaurant serves delicious fresh seafood dishes.",
    "Coffee contains caffeine and is a popular morning drink.",
    "Fresh fruits and vegetables are healthy food choices.",
    "Chocolate cake is a popular dessert for celebrations.",
    
    # Science topic
    "The Earth orbits around the Sun in one year.",
    "Scientists study climate change and global warming.",
    "Physics explains how the universe works at a fundamental level.",
    "Chemistry studies the properties and reactions of substances.",
    "Biology is the study of living organisms and their processes.",
]

print("=== Sample Corpus ===")
print(f"Number of documents: {len(corpus)}")
print(f"\nSample documents:")
for i, doc in enumerate(corpus[:5], 1):
    print(f"  {i}. {doc[:50]}...")

## 2. LDA (Latent Dirichlet Allocation)

LDA is the most popular topic modelling algorithm. It assumes that documents are generated from a mixture of topics, where each topic is a distribution over words.

In [ ]:
# Create document-term matrix using CountVectorizer
count_vectorizer = CountVectorizer(
    max_df=0.95,  # Ignore terms that appear in more than 95% of documents
    min_df=2,     # Ignore terms that appear in fewer than 2 documents
    stop_words='english'
)

doc_term_matrix = count_vectorizer.fit_transform(corpus)
feature_names = count_vectorizer.get_feature_names_out()

print("=== Document-Term Matrix ===")
print(f"Matrix shape: {doc_term_matrix.shape}")
print(f"Vocabulary size: {len(feature_names)}")
print(f"\nVocabulary: {list(feature_names)}")

In [ ]:
# Train LDA model
n_topics = 4

lda_model = LatentDirichletAllocation(
    n_components=n_topics,
    max_iter=20,
    learning_method='online',
    random_state=42,
    n_jobs=-1
)

lda_model.fit(doc_term_matrix)

print("=== LDA Model Training ===")
print(f"Number of topics: {n_topics}")
print(f"Log-likelihood score: {lda_model.score(doc_term_matrix):.2f}")
print(f"Perplexity: {lda_model.perplexity(doc_term_matrix):.2f}")

In [ ]:
# Display topics and their top words
def display_topics(model, feature_names, n_top_words=10):
    """Display topics with their top words."""
    for topic_idx, topic in enumerate(model.components_):
        top_words_idx = topic.argsort()[:-n_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_words_idx]
        top_weights = [topic[i] for i in top_words_idx]
        
        print(f"\nTopic {topic_idx}:")
        print(f"  Words: {', '.join(top_words)}")
        print(f"  Weights: {[f'{w:.2f}' for w in top_weights[:5]]}")

print("=== LDA Topics ===")
display_topics(lda_model, feature_names)

In [ ]:
# Get document-topic distribution
doc_topic_dist = lda_model.transform(doc_term_matrix)

print("=== Document-Topic Distribution ===")
print("\nFirst 5 documents:")
for i in range(min(5, len(corpus))):
    dominant_topic = np.argmax(doc_topic_dist[i])
    print(f"\nDocument {i+1}: {corpus[i][:50]}...")
    print(f"  Dominant topic: {dominant_topic} (prob: {doc_topic_dist[i][dominant_topic]:.4f})")

## 3. NMF (Non-negative Matrix Factorisation)

NMF is an alternative topic modelling algorithm that decomposes the document-term matrix into two non-negative matrices. It often produces more interpretable topics than LDA.

In [ ]:
# Create TF-IDF matrix for NMF
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.95,
    min_df=2,
    stop_words='english'
)

tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

print("=== TF-IDF Matrix for NMF ===")
print(f"Matrix shape: {tfidf_matrix.shape}")

In [ ]:
# Train NMF model
nmf_model = NMF(
    n_components=n_topics,
    random_state=42,
    max_iter=500,
    init='nndsvda'
)

nmf_model.fit(tfidf_matrix)

print("=== NMF Model Training ===")
print(f"Number of topics: {n_topics}")
print(f"Reconstruction error: {nmf_model.reconstruction_err_:.4f}")

In [ ]:
print("=== NMF Topics ===")
display_topics(nmf_model, tfidf_feature_names)

## 4. Comparing LDA and NMF

Let's compare the topics discovered by both algorithms.

In [ ]:
print("=== Comparison: LDA vs NMF ===\n")

print("LDA Topics (top 5 words each):")
for topic_idx, topic in enumerate(lda_model.components_):
    top_words_idx = topic.argsort()[:-6:-1]
    top_words = [feature_names[i] for i in top_words_idx]
    print(f"  Topic {topic_idx}: {', '.join(top_words)}")

print("\nNMF Topics (top 5 words each):")
for topic_idx, topic in enumerate(nmf_model.components_):
    top_words_idx = topic.argsort()[:-6:-1]
    top_words = [tfidf_feature_names[i] for i in top_words_idx]
    print(f"  Topic {topic_idx}: {', '.join(top_words)}")

## 5. Practical Application: Document Clustering

Topic models can be used to cluster documents based on their topic distributions.

In [ ]:
# Cluster documents based on dominant topic
doc_topics_nmf = nmf_model.transform(tfidf_matrix)
dominant_topics = np.argmax(doc_topics_nmf, axis=1)

print("=== Document Clustering by Topic ===\n")

for topic_id in range(n_topics):
    print(f"Topic {topic_id}:")
    topic_docs = [corpus[i] for i in range(len(corpus)) if dominant_topics[i] == topic_id]
    for doc in topic_docs:
        print(f"  - {doc}")
    print()

## 6. Choosing the Number of Topics

Selecting the optimal number of topics is important. We'll use coherence score as a metric.

In [ ]:
# Test different numbers of topics
from sklearn.metrics import silhouette_score

topic_range = range(2, 6)
reconstruction_errors = []

print("=== Finding Optimal Number of Topics ===\n")

for n in topic_range:
    nmf_test = NMF(n_components=n, random_state=42, max_iter=500)
    nmf_test.fit(tfidf_matrix)
    reconstruction_errors.append(nmf_test.reconstruction_err_)
    print(f"Topics: {n}, Reconstruction Error: {nmf_test.reconstruction_err_:.4f}")

print("\nNote: Lower reconstruction error generally indicates better fit,")
print("but you should also consider topic interpretability.")

## Summary

In this notebook, we covered Topic Modelling:

1. **Understanding**: Learned about topic modelling concepts and applications
2. **LDA**: Implemented Latent Dirichlet Allocation for topic discovery
3. **NMF**: Implemented Non-negative Matrix Factorisation as an alternative
4. **Comparison**: Compared LDA and NMF topics
5. **Clustering**: Used topics to cluster documents
6. **Model Selection**: Explored methods for choosing the number of topics

Topic modelling is useful for:
- Document organisation and clustering
- Understanding large document collections
- Information retrieval and search
- Content recommendation systems

LDA tends to produce more probabilistic topics, while NMF often produces more interpretable, non-negative topics. The choice depends on your specific use case.